In [ ]:
import pandas as pd

df = pd.read_csv("synthetic_analytics_survey.csv")

df.head(10)

In [ ]:
import re

pattern = re.compile(r"\b(\d+)\s*\+?\s*(?:years?|yrs?|y)\b", flags=re.IGNORECASE)

In [ ]:
def extract_years(text):
    if pd.isna(text):
        return None
    match = pattern.search(text)
    if match:
        return int(match.group(1))
    return None

df["extracted_yoe"] = df["experience_text"].apply(extract_years)

df[["experience_text", "extracted_yoe"]].head(20)
    

In [ ]:
df.head(10)

In [ ]:
df['preferred_analytics_tool'].unique()

In [ ]:
from rapidfuzz import process, fuzz

ALIASES = {
    "py": "python",
    "structuredquerylanguage": "sql",
    "pbi": "powerbi",
    "js": "javascript"
}

TOOLS = ["python", "sql", "powerbi", "tableau", "excel", "javascript"]

def standardize_tools(text):
    if pd.isna(text):
        return "other"
    text = str(text).lower().replace(" ", "").replace(".", "").replace("-", "") 
    if text in ALIASES:
        return ALIASES[text]
    match = process.extractOne(text, TOOLS, scorer=fuzz.ratio, score_cutoff=80)
    
    return match[0] if match else "other"
    
df["preferred_tool_clean"] = df["preferred_analytics_tool"].apply(standardize_tools)
df[["preferred_analytics_tool", "preferred_tool_clean"]]
print(df["preferred_tool_clean"].value_counts())


In [ ]:
df[df["preferred_tool_clean"] == "other"][["preferred_analytics_tool", "preferred_tool_clean"]]

In [ ]:
df = pd.read_csv("job_postings_messy.csv")
df.head(10)

In [ ]:
import json

raw_value = df.loc[df['company_json'].notna(), 'company_json'].iloc[0]
print(raw_value)
print(type(raw_value))

parsed_value = json.loads(raw_value)
print(parsed_value)
print(type(parsed_value))

In [ ]:
bad_value = df[df['company_json'].fillna("").str.startswith("{'")]['company_json'].iloc[0]
bad_value

In [ ]:
import ast 

def safe_loads(text):
    if pd.isna(text):
        return None
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return ast.literal_eval(text)
    
df['company_parsed'] = df['company_json'].apply(safe_loads)
df[['company_json', 'company_parsed']]

In [ ]:
company_df = pd.json_normalize(df['company_parsed'], sep='_')
company_df.head()

In [ ]:
company_df.index = df.index
df = df.join(company_df)
df.head()

In [ ]:
df['description_html'].iloc[1]

In [ ]:
from bs4 import BeautifulSoup

def html_req_responsibilities_list(description, heading_to_seach_for):
    soup = BeautifulSoup(description, "html.parser")
    h3 = soup.find("h3", string=heading_to_seach_for)
    ul = h3.find_next("ul") if h3 else None
    li = ul.find_all("li") if ul else None
    return [li_item.get_text(strip=True) for li_item in li] if li else []

df['requirements'] = df['description_html'].apply(lambda x: html_req_responsibilities_list(x, 'Requirements'))
df['responsibilities'] = df['description_html'].apply(lambda x: html_req_responsibilities_list(x, 'Responsibilities'))

df[['job_id', 'requirements', 'responsibilities']]

In [ ]:
items = pd.read_csv("order_items.csv")
items.head()

In [ ]:
order_level_df = items.groupby("order_id").agg(
    order_date = ("order_date", "first"),
    customer_id = ("customer_id", "first"),
    total_items = ("quantity", "sum"),
    shipping_cost = ("shipping_cost", "first"),
    order_total = ("order_total", "first")
).reset_index()

order_level_df.head()

In [ ]:
customer_level_df = order_level_df.groupby("customer_id").agg(
num_orders=("order_id", "count"),
total_items=("total_items", "sum"),
total_merchandise=("order_total", "sum"),
first_order=("order_date", "min"),
last_order=("order_date", "max")
).reset_index()

customer_level_df.head()